# Badge 3: Data Application Builders Workshop: Lesson 5: Prototype Reviews & Sprint Planning

NAME_ON_ORDER, the Pending Smoothie Orders app and ORDER_FILLED.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🧬 Mel Presents His Prototype

Reading only.

## Page: 🏃🏽‍♂️‍➡️ Planning Sprint #2

Reading only.

## Page: 📥 Adding a Text Input Box

> **Course:** Move Your Import COL Function Statement to Top of Code (screenshots 2026011415295, 300).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )
''')

> **Course:** Add a Name Box for Smoothie Orders (screenshots 2026011415300.png, 301, 301-1).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

name_on_order = st.text_input('Name on Smoothie:')
st.write('The name on your Smoothie will be:', name_on_order)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients)
            values ('""" + ingredients_string + """')"""

    #st.write(my_insert_stmt)
    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success('Your Smoothie is ordered!', icon="✅")
''')

In [ ]:
%%sql -r run_app_5b
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: ➕ Adding a Column to the Orders Table

> **Course:** Use the ALTER Command to Add a New Column to Your Orders Table: NAME_ON_ORDER (screenshots 2026011415314, 314-1).

In [ ]:
%%sql -r add_name_on_order
-- From screenshot 2026011415314-1
alter table SMOOTHIES.PUBLIC.ORDERS add column name_on_order varchar(100);

## Page: ➕ Writing the Name to the Table

> **Course:** Update the insert statement to include the name and print it with `st.write` and `st.stop()` (screenshots 2026011415324, 325).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

name_on_order = st.text_input('Name on Smoothie:')
st.write('The name on your Smoothie will be:', name_on_order)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients, name_on_order)
            values ('""" + ingredients_string + """','"""+name_on_order+ """')"""

    st.write(my_insert_stmt)
    st.stop()

    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success('Your Smoothie is ordered!', icon="✅")
''')

> **Course:** Uh-oh. Our statement isn't actually valid (screenshot 2026011415325-1, 330). The statement the app printed, run as the course shows it:

In [ ]:
%%sql -r invalid_insert
execute immediate 'begin
  execute immediate ''insert into smoothies.public.orders(ingredients) values (''''Dragon Fruit Honeydew Guava Apples Kiwi '''',''''MellyMel'''')'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

> **Course:** Fix the Insert Statement & Finish the Task: the column list needs `name_on_order` too (screenshot 2026011415330).

In [ ]:
%%sql -r fixed_insert
-- From screenshot 2026011415330
insert into smoothies.public.orders(ingredients, name_on_order) values ('Elderberries Ximenia Ziziphus Jujube Tangerine ','MellyMel');

In [ ]:
%%sql -r select_named_orders
select * from smoothies.public.orders
where name_on_order is not null;

> **Course:** Update your code so the app does the INSERT, and (the fun extra step) add the NAME_ON_ORDER value to the success message (screenshot 2026011415331).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

name_on_order = st.text_input('Name on Smoothie:')
st.write('The name on your Smoothie will be:', name_on_order)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients, name_on_order)
            values ('""" + ingredients_string + """','"""+name_on_order+ """')"""

    # st.write(my_insert_stmt)
    # st.stop()

    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success(f'Your Smoothie is ordered, {name_on_order}!', icon="✅")
''')

In [ ]:
%%sql -r run_app_5d
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

In [ ]:
%%sql -r submit_named_orders
-- Orders submitted through the app (the statement it builds)
insert into smoothies.public.orders(ingredients, name_on_order) values ('Cantaloupe Dragon Fruit Blueberries ','Kristl');
insert into smoothies.public.orders(ingredients, name_on_order) values ('Jackfruit Kiwi Orange Elderberries Yerba Mate ','Gynger');
insert into smoothies.public.orders(ingredients, name_on_order) values ('Kiwi Elderberries Figs Ximenia ','Kai');

## Page: ✅ To-Do #2: A 2nd SiS App for the Kitchen

> **Course:** Build a New SiS App: **Pending Smoothie Orders**, in the SMOOTHIES database and PUBLIC schema. Either create it from scratch or duplicate the existing app and edit it (screenshot 2026011415455).

In [ ]:
%%sql -r create_pending_app
-- SQL equivalent of Duplicate (⋮ → Duplicate): a new app in SMOOTHIES.PUBLIC
use role {{SYSADMIN_ROLE}};
create streamlit SMOOTHIES.PUBLIC."Pending Smoothie Orders"
  runtime_name = 'SYSTEM$WAREHOUSE_RUNTIME'
  query_warehouse = {{WAREHOUSE}}
  title = 'Pending Smoothie Orders';
alter streamlit SMOOTHIES.PUBLIC."Pending Smoothie Orders" add live version from last;

In [ ]:
# a duplicate starts with the original app's code
write_app(PENDING, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

name_on_order = st.text_input('Name on Smoothie:')
st.write('The name on your Smoothie will be:', name_on_order)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients, name_on_order)
            values ('""" + ingredients_string + """','"""+name_on_order+ """')"""

    # st.write(my_insert_stmt)
    # st.stop()

    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success(f'Your Smoothie is ordered, {name_on_order}!', icon="✅")
''')

> **Course:** Add a column named ORDER_FILLED to the ORDERS table: BOOLEAN, `DEFAULT FALSE` right after the word BOOLEAN.

In [ ]:
%%sql -r add_order_filled
-- Derived solution (the course gives the column name, type and default)
alter table SMOOTHIES.PUBLIC.ORDERS add column ORDER_FILLED boolean default false;

> **Course:** Mark some rows as filled before testing the app (the course's code).

In [ ]:
%%sql -r mark_old_orders_filled
update smoothies.public.orders
set order_filled = true
where name_on_order is null;

> **Course:** Delete everything after the `session` and `my_dataframe =` lines and filter to unfilled orders with the course's FILTER code; stop when your app looks like screenshot 2026011415465.

In [ ]:
write_app(PENDING, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

# Write directly to the app
st.title(":cup_with_straw: Pending Smoothie Orders! :cup_with_straw:")
st.write(
    """Orders that need to be filled.
    """
)

session = get_active_session()
my_dataframe = session.table("smoothies.public.orders") \
    .filter(col("ORDER_FILLED") == 0) \
    .collect()
editable_df = st.data_editor(my_dataframe)
''')

## Page: 🖼️ An Editable Dataframe

> **Course:** Convert Your Dataframe to a Data Editor (screenshot 2026011415480): `editable_df = st.data_editor(my_dataframe)`. The first version above already includes that line.

In [ ]:
%%sql -r run_pending_1
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Pending Smoothie Orders"();

## Page: ♻️ Capturing Changes to Data

> **Course:** Add a Submit Button (screenshot 2026011415494).

In [ ]:
write_app(PENDING, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, when_matched

# Write directly to the app
st.title(":cup_with_straw: Pending Smoothie Orders :cup_with_straw:")
st.write("Orders that need to be filled.")

session = get_active_session()
my_dataframe = session.table("smoothies.public.orders") \
    .filter(col("ORDER_FILLED") == 0) \
    .collect()
editable_df = st.data_editor(my_dataframe)

submitted = st.button('Submit')

if submitted:
    st.success("Someone clicked the button.", icon="👍")
''')

In [ ]:
%%sql -r run_pending_2
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Pending Smoothie Orders"();

## Page: 🤖 DORA DABW004

### 🤖 DORA check DABW004 goes here
The course has the learner run **DABW004** (ORDER_FILLED is BOOLEAN with default FALSE) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW004**, as a local check plus the exact DORA code.